In [142]:
# !pip install langchain_google_genai

In [143]:
import os
import re
from google.colab import userdata
from typing import  TypedDict, Annotated, Sequence
import random

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import (BaseMessage, SystemMessage,
                                     AIMessage, HumanMessage)
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [144]:
greetings = [
    "Hey! I'm Vilo. What can I help you with today?",
    "Hi there! 👋 What's on your mind?",
    "Hello! How can I help today?",
    "Hey, I'm Vilo. Feel free to ask me anything.",
    "Hi! What are we working on today?",
    "Hello there! What can I do for you?",
    "Hey! Need a hand with something?",
    "Hi, I'm Vilo. What's up?",
    "Welcome! How can I make things a little easier today?",
    "Hey there! What would you like to chat about?"
]

In [145]:
if not os.environ.get("GEMINI_API_KEY"):
  os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [146]:
llm = ChatGoogleGenerativeAI(
                              model="gemini-2.5-flash",
                              temperature=0.3,
                              # max_output_tokens=300
                             )

In [147]:
# Defining Agent schema
class AgentState(TypedDict):
  messages: Annotated[Sequence[BaseMessage], add_messages]

In [148]:
def answer_user_query(state: AgentState) -> AgentState:
  """
    Processes the user query using the LLM based on the current state.

    Appends a predefined system prompt to the chat history, invokes the
    model, and updates the state with the model's response.

    Args:
        state (AgentState): The current application state containing message history.

    Returns:
        AgentState: The updated state containing the new AI response message.
  """
  print("currently trying to answer user query...")
  system_prompt = SystemMessage(
      content="""
              Your name is vilo, a reliable question-answering visibility logic assistant.

                Your goals are:
                1. Answer the user's question directly.
                2. Be factually accurate.
                3. Be concise by default.
                4. Expand only if the user asks for more detail.
                5. If information is uncertain or unavailable, state that clearly.
                6. Never fabricate facts.
                7. Use Markdown formatting for readability.
              """
  )
  whole_message = [system_prompt] + state["messages"]
  print("Trying to invoke the llm")
  llm_response = llm.invoke(whole_message)
  print(f"llm_response in the answer user query is thus: {llm_response}")
  return {"messages": state["messages"] + [llm_response]}

In [149]:
workflow = StateGraph(AgentState)
workflow.add_node("answer_query_node", answer_user_query)
workflow.add_edge(START, "answer_query_node")
workflow.add_edge("answer_query_node", END)
workflow = workflow.compile()

# workflow.invoke({"messages": "what is kv caching"})

In [150]:
user_history = []
clear_history_count = 0

while True:
  if not user_history:
    print("Vilo: ", random.choice(greetings))
  user_query = input("\nYou: ")

  # To exit the chat by using special keywords "end", "quit", "exit"
  if user_query.lower().strip() in ["end", "quit", "exit"]:
    print("Bye for now! Feel free to come back anytime.")
    break

  normalized = (re.sub(r"[\s_-]+", " ", user_query.lower())).strip()
  if normalized == "clear history":
    clear_history_count += 1
    if clear_history_count > 1:
      print("Chat History already cleared...")
    else:
      print("Clearing Chat History...")
      user_history.clear()

  else:
    clear_history_count = 0 # Resets the clear history count only when its not an special keyword(e.g "exit","clear history", e.t.c)
    user_history.append(HumanMessage(content=user_query)) # stores the HumanMessage object in the user_history(This will serve as chat history)
    response = workflow.invoke({"messages": user_history})  # Send conversation history through the agent and get the updated messages
    user_history = response["messages"] # Updates the user_history with HumanMessage and AIMessage object
    print(user_history)


Vilo:  Hey! Need a hand with something?

You: clear history
Clearing Chat History...
Vilo:  Hey! I'm Vilo. What can I help you with today?

You: clear history
Chat History already cleared...
Vilo:  Hi, I'm Vilo. What's up?

You: quit
Bye for now! Feel free to come back anytime.
